In [1]:
import numpy as np
import os
import cv2
import librosa
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate, TimeDistributed, Conv2D, MaxPooling2D, Flatten

from tensorflow.keras.optimizers import Adam

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
def extract_frames(video_path, num_frames=30):
    try:
        cap = cv2.VideoCapture(video_path)
        frames = []
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_interval = max(total_frames // num_frames, 1)

        for i in range(num_frames):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i * frame_interval)
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (224, 224))
                frames.append(frame)
            else:
                break

        cap.release()
        if len(frames) == 0:
            raise ValueError("No frames extracted")
        return np.array(frames)
    except Exception as e:
        print(f"Error extracting frames from {video_path}: {e}")
        return None

def extract_audio_features(audio_path, n_mfcc=40, max_pad_len=862):
    try:
        y, sr = librosa.load(audio_path, sr=None)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        if mfcc.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfcc.shape[1]
            mfcc = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfcc = mfcc[:, :max_pad_len]
        return mfcc.T

    except Exception as e:
        print(f"Error extracting audio features from {audio_path}: {e}")
        return None

# Paths to dataset
DATASET_PATH = 'E:\\Capstone\\Short Dataset'
CLASS_NAMES = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

video_data = []
audio_data = []
labels = []

for class_name in CLASS_NAMES:
    class_path = os.path.join(DATASET_PATH, class_name)
    for video_name in os.listdir(class_path):
        video_path = os.path.join(class_path, video_name)
        if not video_name.endswith('.mp4'):
            continue
        frames = extract_frames(video_path)
        audio_features = extract_audio_features(video_path)

        if frames is not None and audio_features is not None:
            video_data.append(frames)
            audio_data.append(audio_features)
            labels.append(CLASS_NAMES.index(class_name))
        else:
            print(f"Skipping {video_name} due to insufficient data")

# Pad video sequences to a uniform length
video_data_padded = pad_sequences(video_data, padding='post', dtype='float32')
audio_data_padded = pad_sequences(audio_data, padding='post', dtype='float32')
labels = np.array(labels)

print("Video data shape:", video_data_padded.shape)
print("Audio data shape:", audio_data_padded.shape)
print("Labels shape:", labels.shape)

C:\Users\ehsan\AppData\Local\Temp\ipykernel_6028\966810384.py:27: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=None)
c:\Python10\lib\site-packages\librosa\core\audio.py:183: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
C:\Users\ehsan\AppData\Local\Temp\ipykernel_6028\966810384.py:27: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=None)
c:\Python10\lib\site-packages\librosa\core\audio.py:183: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Video data shape: (105, 30, 224, 224, 3)
Audio data shape: (105, 862, 40)
Labels shape: (105,)


In [3]:
# Pad video sequences to a uniform length
video_data_padded = pad_sequences(video_data, padding='post', dtype='float32')
labels = np.array(labels)

print("Video data shape:", video_data_padded.shape)
print("Labels shape:", labels.shape)

Video data shape: (105, 30, 224, 224, 3)
Labels shape: (105,)


In [6]:
# One-hot encode labels
labels_categorical = to_categorical(labels, num_classes=len(CLASS_NAMES))

# Video model
video_input = Input(shape=(video_data_padded.shape[1], 224, 224, 3))
x = TimeDistributed(Conv2D(32, (3, 3), activation='relu'))(video_input)
x = TimeDistributed(MaxPooling2D((2, 2)))(x)
x = TimeDistributed(Flatten())(x)
x = LSTM(64, return_sequences=False)(x)
x = Dropout(0.5)(x)
video_output = Dense(len(CLASS_NAMES), activation='softmax')(x)

video_model = Model(inputs=video_input, outputs=video_output)
audio_model.compile(optimizer=Adam(lr=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

video_model.summary()

NameError: name 'audio_model' is not defined

In [5]:
# Train the video model
video_model.fit(
    video_data_padded,
    labels_categorical,
    epochs=50,
    batch_size=8,
    validation_split=0.3,
    shuffle=True
)

Epoch 1/50




InvalidArgumentError: Graph execution error:

Detected at node sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/SparseSoftmaxCrossEntropyWithLogits defined at (most recent call last):
  File "c:\Python10\lib\runpy.py", line 196, in _run_module_as_main

  File "c:\Python10\lib\runpy.py", line 86, in _run_code

  File "C:\Users\ehsan\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "C:\Users\ehsan\AppData\Roaming\Python\Python310\site-packages\traitlets\config\application.py", line 1075, in launch_instance

  File "C:\Users\ehsan\AppData\Roaming\Python\Python310\site-packages\ipykernel\kernelapp.py", line 739, in start

  File "C:\Users\ehsan\AppData\Roaming\Python\Python310\site-packages\tornado\platform\asyncio.py", line 205, in start

  File "c:\Python10\lib\asyncio\base_events.py", line 600, in run_forever

  File "c:\Python10\lib\asyncio\base_events.py", line 1896, in _run_once

  File "c:\Python10\lib\asyncio\eve

In [ ]:
# Save the video model
video_model.save('/content/drive/MyDrive/video_model.h5')

In [ ]:
import numpy as np
import os
import librosa
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [ ]:
# Pad audio sequences to a uniform length
audio_data_padded = pad_sequences(audio_data, padding='post', dtype='float32')
labels = np.array(labels)

print("Audio data shape:", audio_data_padded.shape)
print("Labels shape:", labels.shape)

# One-hot encode labels
labels_categorical = to_categorical(labels, num_classes=len(CLASS_NAMES))

# Audio model
audio_input = Input(shape=(audio_data_padded.shape[1], audio_data_padded.shape[2]))
y = LSTM(128, return_sequences=False)(audio_input)
y = Dropout(0.5)(y)
audio_output = Dense(len(CLASS_NAMES), activation='softmax')(y)

audio_model = Model(inputs=audio_input, outputs=audio_output)
audio_model.compile(optimizer=Adam(lr=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

audio_model.summary()


In [ ]:
# Train the audio model
audio_model.fit(
    audio_data_padded,
    labels_categorical,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    shuffle=True
)

In [ ]:
# Save the audio model
audio_model.save('/content/drive/MyDrive/audio_model.h5')

In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Average
from tensorflow.keras.models import Model

# Load trained models
video_model = load_model('/content/drive/MyDrive/video_model.h5')
audio_model = load_model('/content/drive/MyDrive/audio_model.h5')

# Ensure the models' outputs are probability distributions
video_output = video_model.output
audio_output = audio_model.output

# Late fusion
combined_output = Average()([video_output, audio_output])

In [ ]:
# Define the combined model
combined_model = Model(inputs=[video_model.input, audio_model.input], outputs=combined_output)
combined_model.compile(optimizer=Adam(lr=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

combined_model.summary()

In [ ]:
# One-hot encode test labels
labels_test_categorical = to_categorical(labels, num_classes=len(CLASS_NAMES))

In [ ]:
# Evaluate the combined model
combined_model.evaluate([video_data, audio_data], labels_test_categorical)